# Test du pipeline d'ingestion

Le but est de vérifier que le pipeline `src/ingest/` produit **les mêmes chunks** que le notebook d'exploration.

1. Sauvegarde de la sortie du notebook d'exploration (référence)
2. Exécution du pipeline
3. Contrôles rapides sur les chunks
4. Comparaison avec la référence (ids, texte, métadonnées)

In [2]:
import json
import logging
import shutil

import pandas as pd

from assistant_regles.ingest.chunk import lire_jsonl
from assistant_regles.ingest.config import charger_config
from assistant_regles.ingest.pipeline import executer

logging.basicConfig(level=logging.INFO, format="%(levelname)-7s %(name)s — %(message)s", force=True)

config = charger_config()  # trouve la racine du repo depuis notebooks/
print("PDF   :", config.chemin_pdf, "| existe :", config.chemin_pdf.exists())
print("Cache :", config.chemin_cache_docling, "| existe :", config.chemin_cache_docling.exists())

PDF   : /home/saucisse-de-sanglier/dev/assistant-regles-w40k/data/core-rules/fr-warhammer40k_regles_de_base_01_06_2026.pdf | existe : True
Cache : /home/saucisse-de-sanglier/dev/assistant-regles-w40k/data/interim/conversion/fr-warhammer40k_regles_de_base_docling.json | existe : True


## 1. Référence : sortie du notebook d'exploration
Le pipeline écrit au même endroit : on copie d'abord l'ancien fichier pour pouvoir comparer.

In [3]:
reference = config.chemin_chunks.with_name("chunks_notebook.jsonl")
if config.chemin_chunks.exists() and not reference.exists():
    shutil.copy(config.chemin_chunks, reference)
print("Référence disponible :", reference.exists())

Référence disponible : True


In [9]:
config.chemin_chunks

PosixPath('/home/saucisse-de-sanglier/dev/assistant-regles-w40k/data/interim/chunks.jsonl')

## 2. Exécution du pipeline
Recharge le cache Docling s'il existe (sinon conversion complète, plusieurs minutes).

In [4]:
resultat = executer(config)
resultat

INFO    assistant_regles.ingest.parse — Conversion Docling rechargée depuis le cache fr-warhammer40k_regles_de_base_docling.json
INFO    assistant_regles.ingest.parse — 1321 éléments de contenu, 319 repères structurels
INFO    assistant_regles.ingest.pipeline — Codes présents plusieurs fois : {'15.11': 2}
INFO    assistant_regles.ingest.pipeline — Nettoyage :
motif_exclusion
conserve            997
etiquette_image     199
renvoi               47
numero_section       30
zone_hors_regles     18
lore_stratageme      10
doublon               9
cout_isole            9
sans_contenu          2
/home/saucisse-de-sanglier/dev/assistant-regles-w40k/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
INFO    httpx — HTTP Request: HEAD https://huggingface.co/BAAI/bge-m3/resolve/main/config.json "HTTP/1.1 307 Temp

ResultatIngestion(nb_elements=1321, nb_elements_conserves=997, nb_chunks=207, chemin_elements=PosixPath('/home/saucisse-de-sanglier/dev/assistant-regles-w40k/data/interim/elements.parquet'), chemin_chunks=PosixPath('/home/saucisse-de-sanglier/dev/assistant-regles-w40k/data/interim/chunks/chunks.jsonl'))

## 3. Contrôles rapides

In [5]:
chunks = lire_jsonl(resultat.chemin_chunks)
df_chunks = pd.DataFrame([c.model_dump() for c in chunks])

print(df_chunks["nb_tokens"].describe(percentiles=[.5, .9, .95]).round(0))
print("Hors budget          :", df_chunks["hors_budget"].sum())
print("Unités découpées     :", ((df_chunks["partie"] == 1) & (df_chunks["nb_parties"] > 1)).sum())
print("Identifiants uniques :", df_chunks["id"].is_unique)

# Couverture : chaque élément conservé apparaît dans au moins un chunk
df_elements = pd.read_parquet(resultat.chemin_elements)
conserves = set(df_elements.loc[df_elements["motif_exclusion"].isna(), "ordre"])
couverts = {o for c in chunks for o in c.ordres}
print("Éléments non couverts :", len(conserves - couverts))

count    207.0
mean     210.0
std      132.0
min       36.0
50%      181.0
90%      391.0
95%      454.0
max      568.0
Name: nb_tokens, dtype: float64
Hors budget          : 0
Unités découpées     : 11
Identifiants uniques : True
Éléments non couverts : 0


In [6]:
print(df_chunks.sample(1, random_state=1)["texte"].iloc[0])

RÉFÉRENCES > 24 Aptitudes de base > 24.35 MARCHEUR SUPER-LOURD

Créatures monstrueuses et machines de guerre gigantesques surplombent le champ de bataille comme des dieux manifestés, qui enjambent les combattants pour écraser tous les obstacles.
Chaque fois qu'une unité avec cette aptitude effectue un mouvement normal, d'avance ou de retraite :
Les figurines de cette unité peuvent se déplacer à travers les figurines (y compris les figurines de MONSTRE/ VÉHICULE, mais figurines TITANESQUES exclues) et peuvent se déplacer à l'horizontale à travers des sections d'éléments de terrain qui font 4' ou moins de hauteur.
Avant de déplacer cette unité, vous pouvez décider que toutes les figurines de l'unité aient le mot-clé MOBILE jusqu'à la fin de ce mouvement. Dans ce cas, quand ce mouvement se termine, jetez 1 D6 : sur 1, l'unité est ébranlée.
Note de Conception : Gagner le mot-clé MOBILE pour la durée d'un mouvement permettra à cette unité de se déplacer horizontalement à travers les élément

## 4. Comparaison avec la référence
Objectif : 0 id manquant de chaque côté et 0 différence.

In [8]:
def charger_par_id(chemin):
    with open(chemin, encoding="utf-8") as f:
        return {d["id"]: d for d in map(json.loads, f)}

ref, nouv = charger_par_id(reference), charger_par_id(resultat.chemin_chunks)
print("Chunks notebook / pipeline :", len(ref), "/", len(nouv))
print("Ids seulement dans le notebook :", len(ref.keys() - nouv.keys()))
print("Ids seulement dans le pipeline :", len(nouv.keys() - ref.keys()))

CHAMPS = ["texte", "code", "sous_section", "page_debut", "page_fin", "codes_cites", "nb_tokens"]
differences = [
    {"id": i, "champ": champ, "notebook": ref[i][champ], "pipeline": nouv[i][champ]}
    for i in sorted(ref.keys() & nouv.keys())
    for champ in CHAMPS
    if ref[i][champ] != nouv[i][champ]
]
print("Différences sur les chunks communs :", len(differences))
pd.DataFrame(differences).head(10)

Chunks notebook / pipeline : 207 / 207
Ids seulement dans le notebook : 0
Ids seulement dans le pipeline : 0
Différences sur les chunks communs : 0


""
